In [1]:
import sqlite3
import json
import os
import h5py

In [ ]:
class ResearchSql():
    def __init__(self, base_dir, db_name):
        self.conn = sqlite3.connect(db_name)
        self.conn.row_factory = sqlite3.Row # row_factory 使查询结果可以像字典一样访问
        self.cursor = self.conn.cursor()
        self._init_table_struture()
        # self._init_table_content(base_dir)

    def _init_table_struture(self):
        query = '''
        CREATE TABLE IF NOT EXISTS pipeline_info (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            category_name TEXT,
            instance_name TEXT,
            frame_idx TEXT,
            clustered_img_path TEXT,
            status TEXT,
            vlm_response TEXT,
            error_msg TEXT,
            UNIQUE(category_name, instance_name, frame_idx)
        )
        '''
        self.cursor.execute(query)
        self.conn.commit()

    def _init_table_content(self, base_dir):
        h5_paths = []
        for name in os.listdir(base_dir):
            path = os.path.join(base_dir, name)
            if not os.path.isfile(path):
                continue
            if not name.endswith('.h5'):
                continue
            category_name = name.split('.')[0]
            with h5py.File(path, 'r') as h5_file:
                instance_keys = list(h5_file.keys()) # chair1, chair2...
                for instance_name in instance_keys:
                    self.add_data(category_name, instance_name, '', '', '', '', '')

    def add_data(self, category_name, instance_name, frame_idx, clustered_img_path, status, vlm_response, error_msg):
        query = '''
        INSERT INTO pipeline_info (category_name, instance_name, frame_idx, clustered_img_path, status, vlm_response, error_msg)
        VALUES (?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(category_name, instance_name, frame_idx) 
        DO UPDATE SET 
            clustered_img_path = excluded.clustered_img_path,
            status = excluded.status,
            error_msg = excluded.error_msg
        '''
        self.cursor.execute(query, (category_name, instance_name, frame_idx, clustered_img_path, status, vlm_response, error_msg))
        self.conn.commit()

    def update_content(self, category_name, instance_name, frame_idx,  
                       clustered_img_path=None, status=None, vlm_response=None, error_msg=None):
        update_fields = []
        update_values = []

        if frame_idx is not None:
            update_fields.append('frame_idx = ?')
            update_values.append(frame_idx)

        if clustered_img_path is not None:
            update_fields.append('clustered_img_path = ?')
            update_values.append(clustered_img_path)

        if status is not None:
            update_fields.append('status = ?')
            update_values.append(status)

        if vlm_response is not None:
            update_fields.append('vlm_response = ?')
            update_values.append(vlm_response)

        if error_msg is not None:
            update_fields.append('error_msg = ?')
            update_values.append(error_msg)

        if not update_fields:
            raise ValueError('No content provided to update.')

        query = f'''
        UPDATE pipeline_info
        SET {', '.join(update_fields)}
        WHERE category_name = ? AND instance_name = ? AND frame_idx = ?
        '''

        update_values.extend([category_name, instance_name])
        self.cursor.execute(query, tuple(update_values))
        self.conn.commit()

    def get_instance_data(self, instance_name):
        query = '''
        SELECT *
        FROM pipeline_info
        WHERE instance_name = ?
        ORDER BY frame_idx ASC
        '''
        self.cursor.execute(query, (instance_name,))
        rows = self.cursor.fetchall()
        result = [dict(row) for row in rows]
        return result

    def delete_data(self, category_name, instance_name, frame_idx=''):
        query = '''
        DELETE FROM pipeline_info
        WHERE category_name = ? AND instance_name = ? AND frame_idx = ?
        '''
        self.cursor.execute(query, (category_name, instance_name, frame_idx))
        self.conn.commit()
        
    def close(self):
        self.conn.close()

In [21]:
base_dir = '/root/autodl-tmp/ARLP/dataset/h5'
db_name = '/root/autodl-tmp/ARLP/dataset/tests/test.sql'
if os.path.exists(db_name):
    pass
else:
    db = ResearchSql(base_dir, db_name)

# 获取 JSON 数据
data = db.get_instance_data('zaihmr')
print(f'data type', type(data))
print(data)
print(data[0])
print(data[0]['instance_name'])

data type <class 'list'>
[{'id': 87, 'category_name': 'carving_knife', 'instance_name': 'zaihmr', 'frame_idx': '', 'clustered_img_path': '', 'status': '', 'vlm_response': '', 'error_msg': ''}]
{'id': 87, 'category_name': 'carving_knife', 'instance_name': 'zaihmr', 'frame_idx': '', 'clustered_img_path': '', 'status': '', 'vlm_response': '', 'error_msg': ''}
zaihmr


In [23]:
category_name = 'whisk'
instance_name = 'ovaccb'
db.update_content(category_name, instance_name, frame_idx = 1)
frame_idx = db.get_instance_data(instance_name)[0]['frame_idx']
print(frame_idx)

1


In [24]:
instance_name = 'ovaccb'
top_k_indices = []
instance_rows = db.get_instance_data(instance_name) # [{'name1':'name1, 'topk': topk..},{.}, {.}..]

for i in range(len(instance_rows)):
    top_k_indices.append(int(instance_rows[i]['frame_idx']))
print(top_k_indices)

[1]
